In [0]:
# Set the Delta path (Volume-safe)
DELTA_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events"

In [0]:
# Load DeltaTable and inspect history
from delta.tables import DeltaTable
from pyspark.sql import functions as F

dt = DeltaTable.forPath(spark,DELTA_PATH)

# version history (proves time travel capability exists)
dt.history(10).select("version","timestamp","operation","operationParameters").show(truncate=False)

In [0]:
# Implement incremental MERGE
## Read current snapshot
base = spark.read.format("delta").load(DELTA_PATH)
base.printSchema()
print ("Base rows:",base.count())

In [0]:
## Create realistic incremental updates
### - Take a small sample of existing rows and update a numeric column (price)
### - Create a few new rows that should insert

updates_existing = (
    base
    .filter(F.col("price").isNotNull())
    .select(*base.columns)
    .limit(100)
    .withColumn("price",F.col("price")+F.lit(1.0).cast("double"))
)

# Create "new" rows (insert candidates) by tweaking keys slightly
# We'll add a synthetic batch_id and ingest_ts (architect-level audit trail)

updates_new = (
    updates_existing
    .limit(50)
    .withColumn("user_id", (F.col("user_id") + F.lit(999999)).cast("long"))
)

batch_id = "day05_batch_001"

updates = (
    updates_existing.unionByName(updates_new)
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingest_ts", F.current_timestamp())
)
print("Updates rows:", updates.count())

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Determine key columns from what exists
if "user_session" in updates.columns and "event_time" in updates.columns:
    key_cols = ["user_session", "event_time"]
elif "user_id" in updates.columns and "product_id" in updates.columns and "event_time" in updates.columns:
    key_cols = ["user_id", "product_id", "event_time"]
elif "user_id" in updates.columns and "product_id" in updates.columns and "event_ts" in updates.columns:
    key_cols = ["user_id", "product_id", "event_ts"]
else:
    raise Exception(f"Cannot dedupe: no suitable key columns found in updates. Columns: {updates.columns}")

# Keep latest record per key based on ingest_ts
w = Window.partitionBy(*key_cols).orderBy(F.col("ingest_ts").desc())

updates_dedup = (
    updates
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Updates before:", updates.count())
print("Updates after :", updates_dedup.count())

# MERGE using deduped updates
(
    dt.alias("t")
    .merge(updates_dedup.alias("s"), merge_condition)
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


In [0]:
dt.history(5).select("version", "timestamp", "operation").show(truncate=False)

In [0]:
# Time travel
## - Read by version
latest_version = dt.history(1).select("version").collect()[0][0]
v0 = spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH)
v_latest = spark.read.format("delta").option("versionAsOf", latest_version).load(DELTA_PATH)

print("v0 rows:", v0.count())
print("latest rows:", v_latest.count())
print("latest version:", latest_version)

In [0]:
# Optimize + ZORDER
try:
    spark.sql("OPTIMIZE events_table ZORDER BY (event_type, user_id)")
    print("OPTIMIZE/ZORDER succeeded.")
except Exception as e:
    print("OPTIMIZE/ZORDER not available in this environment:")
    print(e)

In [0]:
# VACUUM cleanup

try:
    spark.sql("VACUUM events_table RETAIN 168 HOURS")
    print("VACUUM succeeded.")
except Exception as e:
    print("VACUUM not available or restricted in this environment:")
    print(e)

In [0]:
# Add idempotency + audit pattern
## - Dedupe updates before merge

# If duplicate records arrive in the same batch, keep the latest by ingest_ts
# (real world: this prevents merge churn and non-determinism)

updates_dedup = updates
if "ingest_ts" in updates.columns:
    if "user_session" in updates.columns and "event_time" in updates.columns:
        w = Window.partitionBy("user_session", "event_time").orderBy(F.col("ingest_ts").desc())
    elif "user_id" in updates.columns and "product_id" in updates.columns and ("event_time" in updates.columns or "event_ts" in updates.columns):
        time_col = "event_time" if "event_time" in updates.columns else "event_ts"
        w = Window.partitionBy("user_id", "product_id", time_col).orderBy(F.col("ingest_ts").desc())
    else:
        w = None

    if w:
        updates_dedup = (
            updates_dedup
            .withColumn("rn", F.row_number().over(w))
            .filter(F.col("rn") == 1)
            .drop("rn")
        )

print("Updates after dedupe:", updates_dedup.count())


In [0]:
## - Merge using deduped updates
dt.alias("t").merge(
    updates_dedup.alias("s"),
    merge_condition
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

dt.history(5).select("version", "timestamp", "operation").show(truncate=False)
